# Thai Doc Classifier — run on a free Kaggle GPU (git pull)

Clones your repo and runs `bench_local.py` on Kaggle's free GPU.

## Setup in the right sidebar BEFORE running
1. **Accelerator → `GPU T4 x2`** ⚠️ **NOT `GPU P100`** — Kaggle's current PyTorch dropped support for the P100 (sm_60); only T4 (sm_75) and newer work. A P100 gives `no kernel image is available`.
2. **Internet → On**.
3. **Private repo?** Add-ons → **Secrets** → add `GITHUB_TOKEN` = a GitHub PAT (`repo` read scope).
4. Set `REPO_URL` in cell 1.

## Two ways to run
- **3B in fp16** — no bitsandbytes, always works on a 16 GB T4. Best first run.
- **7B in 4-bit** — needs bitsandbytes (works on T4, not P100).

## 0. Check the GPU (must be T4 or newer)

In [ ]:
import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv
cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
print('torch', torch.__version__, '| GPU', name, '| capability', cap)
if cap[0] < 7:
    print(f'\n*** WARNING: {name} (sm_{cap[0]}{cap[1]}) is NOT supported by this torch. ***')
    print('*** Sidebar -> Accelerator -> GPU T4 x2 (not P100), then Restart. ***')
else:
    print('GPU supported - good to go.')

## 1. Clone your repo
Set `REPO_URL`. For a **private** repo, store a PAT as the `GITHUB_TOKEN` secret (injected without being printed).

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/<you>/<repo>.git'   # <-- your repo
DST = '/kaggle/working/project'

url = REPO_URL
try:  # private repo: inject token from Kaggle Secrets (not printed)
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret('GITHUB_TOKEN')
    url = REPO_URL.replace('https://', f'https://{tok}@')
except Exception:
    pass  # public repo or no secret -> plain clone

if not os.path.exists(DST):
    r = subprocess.run(['git', 'clone', '--depth', '1', url, DST])
    if r.returncode != 0:
        raise SystemExit('git clone failed - check REPO_URL / GITHUB_TOKEN / Internet')
os.chdir(DST)
print('cwd:', os.getcwd(), '| has bench_local.py:', os.path.exists('bench_local.py'))

## 2. Install dependencies (safely)
**Do NOT upgrade torch** — Kaggle's torch matches its GPUs. We upgrade only the model libs and install the **latest** bitsandbytes (its cu128 binary supports the T4; older pins hit a `triton.ops` error).

In [ ]:
!pip install -q -U transformers accelerate qwen-vl-utils openpyxl pillow \
    opencv-python-headless scikit-learn matplotlib
!pip install -q -U bitsandbytes   # latest (needed only for 4-bit / the 7B)
print('deps installed')

## 2b. Verify CUDA works before downloading a big model

In [ ]:
import torch
try:
    print('torch cuda op OK:', (torch.randn(8, device='cuda') * 2).sum().item())
except Exception as e:
    print('TORCH CUDA BROKEN ->', e)
    print('=> Almost certainly a P100. Switch Accelerator to GPU T4 x2 and Restart.')
try:
    import bitsandbytes as bnb
    print('bitsandbytes', bnb.__version__)
except Exception as e:
    print('bitsandbytes import failed ->', e, '(fine if you only use the 3B fp16 path)')

## 3. Point at test images
`test-files/` is gitignored, so it's not in the clone. Upload your images as a **Kaggle Dataset** (Input → + Add Input → Upload) and set `DATA_DIR` to it, e.g. `/kaggle/input/thaidoc-test-files`.

In [ ]:
import os

DATA_DIR = 'test-files'                       # or '/kaggle/input/<your-dataset>'
exts = ('.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp', '.webp')

if not os.path.isdir(DATA_DIR) or not [f for f in os.listdir(DATA_DIR) if f.lower().endswith(exts)]:
    print('No images in', DATA_DIR, '- set DATA_DIR to /kaggle/input/<dataset>,')
    print('or generate synthetic:  from thaidoc.synth import generate; generate(2); DATA_DIR="data/synth/images"')

imgs = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(exts)] if os.path.isdir(DATA_DIR) else []
print('images:', len(imgs), 'in', DATA_DIR)
for f in sorted(imgs):
    print(' -', f)

## 4a. Run the 3B in fp16 — no bitsandbytes (always works on a 16 GB T4)

In [ ]:
import os
os.environ['THAIDOC_LLM_4BIT'] = '0'   # fp16, skips bitsandbytes
!python bench_local.py --provider transformers \
    --model Qwen/Qwen2.5-VL-3B-Instruct --dir "$DATA_DIR"

## 4b. Run the 7B in 4-bit — needs bitsandbytes (T4 only)
Run only if cell 2b showed `torch cuda op OK` + a bitsandbytes version. The 7B in fp16 won't fit a 16 GB T4, so it needs 4-bit.

In [ ]:
import os
os.environ['THAIDOC_LLM_4BIT'] = 'auto'   # 4-bit via bitsandbytes
!python bench_local.py --provider transformers \
    --model Qwen/Qwen2.5-VL-7B-Instruct --dir "$DATA_DIR"

## 5. View the text report

In [ ]:
print(open('bench_report_transformers.txt', encoding='utf-8').read())